# 02 · Clean & Audit — Peras / PostHog

**Purpose:** Turn the raw extract from `01_extract.ipynb` into something trustworthy —
dedupe, sanity-check nulls/types, strip bot & non-production traffic, define sessions,
and summarize it all in a short data-quality report.

**Safety contract for this notebook**
- Reads `data/raw/events_raw.parquet` (private tier, gitignored) — no PostHog
  credentials needed here, this is a local pandas/SQL pass over the extract.
- Writes `data/raw/events_clean.parquet` — still private tier, still gitignored.
  This is *not* the public/aggregated tier; no anonymization or small-cell
  suppression happens here, that's downstream in `03_eda.ipynb`.
- `reports/data_quality.md` is **not** gitignored — keep it to pipeline-health
  metrics (dedup rate, bot rate, null rates, session stats), not sensitive
  business figures (revenue, absolute engagement counts), per `CLAUDE.md`'s
  safety rules.

**Before running:** run `01_extract.ipynb` first so `data/raw/events_raw.parquet` exists.


In [ ]:
# Dependencies (run once). Safe to re-run.
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
REPORTS_DIR = Path("../reports")
IN_PATH = RAW_DIR / "events_raw.parquet"

assert IN_PATH.exists(), f"{IN_PATH} not found -- run 01_extract.ipynb first."

events = pd.read_parquet(IN_PATH)
# ISO8601 explicitly -- PostHog returns mixed fractional-second precision
# (some rows have microseconds, some don't), which trips plain to_datetime().
events["timestamp"] = pd.to_datetime(events["timestamp"], utc=True, format="ISO8601")

n_raw = len(events)
print(f"Loaded {n_raw:,} rows, {events['event'].nunique()} event types, "
      f"{events['person_id'].nunique():,} unique persons.")
events.dtypes

Loaded 114,377 rows, 174 event types, 4,558 unique persons.


event                                        str
timestamp                    datetime64[us, UTC]
distinct_id                                  str
person_id                                    str
account_id                                   str
project_id                                   str
current_url                                  str
browser                                      str
os                                           str
country                                      str
referring_domain                             str
entry_path                                   str
auth_provider                                str
utm_source                                   str
utm_medium                                   str
utm_campaign                                 str
utm_term                                     str
landing_experiment                           str
landing_variant                              str
campaign_intent                              str
campaign_variant    

## Step 1 · Structural checks (nulls & types)

Most of the columns in this extract are event-specific properties (e.g. `project_id`
only exists on project/generation events, `reader_section_count` only on reader
events) — a high null rate on those is expected, not a bug. What actually matters:
the small set of **core** columns that should be populated on every row, and that
numeric/boolean columns came through as the right dtype rather than object/string.


In [2]:
null_rates = events.isna().mean().sort_values(ascending=False).to_frame("null_rate")
null_rates["null_pct"] = (null_rates["null_rate"] * 100).round(1)
null_rates.drop(columns="null_rate")

,null_pct
failure_reason,100.0
generation_mode,99.7
project_id,99.4
auth_provider,99.2
reader_written_sections,99.1
utm_term,99.1
reader_section_count,99.0
reader_mode,98.6
has_paced_reader,98.6
account_id,97.1


In [3]:
# Core columns that should never be null -- these identify and place every event.
core_cols = ["event", "timestamp", "distinct_id", "person_id"]

for col in core_cols:
    n_missing = events[col].isna().sum()
    print(f"{col}: {n_missing:,} missing ({n_missing / n_raw:.2%})")

assert events[core_cols].notna().all().all(), "Core identity/timestamp columns should never be null"
print("OK: all core columns fully populated.")

event: 0 missing (0.00%)
timestamp: 0 missing (0.00%)
distinct_id: 0 missing (0.00%)
person_id: 0 missing (0.00%)
OK: all core columns fully populated.


## Step 2 · Deduplication

Note: this extract doesn't include a per-event unique id (PostHog's `uuid` or
`$insert_id`) — add one to `01_extract.ipynb`'s column list if exact dedup ever
matters more than this. For now this catches **exact duplicate rows** across every
column we did pull, which is enough to catch double-fired client events.


In [4]:
dupe_mask = events.duplicated(keep="first")
n_dupes = int(dupe_mask.sum())
print(f"Exact duplicate rows: {n_dupes:,} ({n_dupes / n_raw:.2%})")

events_deduped = events.loc[~dupe_mask].reset_index(drop=True)
n_deduped = len(events_deduped)
print(f"Rows after dedup: {n_deduped:,} (removed {n_raw - n_deduped:,})")

Exact duplicate rows: 1,073 (0.94%)
Rows after dedup: 113,304 (removed 1,073)


## Step 3 · Bot & non-production traffic

Confirmed via PostHog's schema before writing this filter (don't guess at values):
`environment` is tagged `"production"` when set; `traffic_type` (from
`$virt_traffic_type`) has more categories than just bot/not-bot -- `"Regular"`,
`"Bot"`, `"AI Agent"`, `"Automation"`, and others. Only `"Regular"` is real human
product usage.

Both properties are set by newer SDK/server versions, so some historical rows may
have them as null rather than an explicit value. Treating null as "assume real" (not
"assume bad") avoids silently dropping legitimate older/server-side events that were
never tagged -- we only drop rows *explicitly* marked bot or non-production.


In [5]:
print("environment value counts:")
print(events_deduped["environment"].value_counts(dropna=False))
print()
print("traffic_type value counts:")
print(events_deduped["traffic_type"].value_counts(dropna=False))

environment value counts:
environment
production    100032
NaN            13251
prod              21
Name: count, dtype: int64

traffic_type value counts:
traffic_type
Regular       103992
Automation      8958
Bot              354
Name: count, dtype: int64


In [6]:
before = len(events_deduped)

non_prod_mask = events_deduped["environment"].notna() & (events_deduped["environment"] != "production")
bot_mask = events_deduped["traffic_type"].notna() & (events_deduped["traffic_type"] != "Regular")

# Cross-check: is_bot vs traffic_type should agree. Flag if not -- worth a closer look,
# not something to silently paper over.
is_bot_flag = events_deduped["is_bot"].fillna(False).astype(bool)
mismatch = is_bot_flag != (events_deduped["traffic_type"] == "Bot")
n_mismatch = int(mismatch.sum())
if n_mismatch:
    print(f"NOTE: {n_mismatch:,} rows where is_bot and traffic_type disagree -- inspect before trusting either alone.")

events_clean = events_deduped.loc[~non_prod_mask & ~bot_mask].reset_index(drop=True)
n_clean = len(events_clean)

n_non_prod = int(non_prod_mask.sum())
n_bot = int(bot_mask.sum())
print(f"Non-production rows removed: {n_non_prod:,} ({n_non_prod / before:.2%})")
print(f"Bot/AI-agent/automation rows removed: {n_bot:,} ({n_bot / before:.2%})")
print(f"Rows after traffic filtering: {n_clean:,} (from {before:,})")

NOTE: 8,958 rows where is_bot and traffic_type disagree -- inspect before trusting either alone.
Non-production rows removed: 21 (0.02%)
Bot/AI-agent/automation rows removed: 9,312 (8.22%)
Rows after traffic filtering: 103,992 (from 113,304)


## Step 4 · Session definition

PostHog already assigns a client-side `session_id` (~30 min inactivity timeout) --
that's the session definition we use here rather than re-deriving one, since it's
already computed and consistent with how PostHog's own web analytics/session replay
count sessions. Some server-only events won't carry a `session_id` (no browser
session context); those are reported separately rather than forced into one.

Aggregation done in DuckDB (per the project's toolchain: local SQL over the
extracts) rather than a pandas groupby, to keep the SQL/Python split visible.


In [7]:
import duckdb

with_session = events_clean["session_id"].notna().sum()
print(f"Rows with a session_id: {with_session:,} ({with_session / n_clean:.2%})")
print(f"Rows with no session context (server-side events): {n_clean - with_session:,}")

con = duckdb.connect()
session_stats = con.execute("""
    SELECT
        session_id,
        person_id,
        count(*)                                             AS event_count,
        min(timestamp)                                        AS session_start,
        max(timestamp)                                        AS session_end,
        date_diff('second', min(timestamp), max(timestamp))   AS duration_seconds
    FROM events_clean
    WHERE session_id IS NOT NULL
    GROUP BY session_id, person_id
""").df()

n_sessions = len(session_stats)
median_events_per_session = session_stats["event_count"].median()
median_session_seconds = session_stats["duration_seconds"].median()

print(f"Sessions: {n_sessions:,}")
print(f"Median events/session: {median_events_per_session:.1f}")
print(f"Median session duration: {median_session_seconds:.0f}s")
# Aggregate distribution only -- never preview raw session_id/person_id rows here.
# This notebook (and its saved outputs) gets committed even though data/raw/ itself
# is gitignored.
session_stats[["event_count", "duration_seconds"]].describe()

Rows with a session_id: 103,992 (100.00%)
Rows with no session context (server-side events): 0
Sessions: 5,171
Median events/session: 12.0
Median session duration: 6s


,event_count,duration_seconds
count,5171.000000,5171.000000
mean,20.110617,161.106749
std,60.767599,732.161738
min,1.000000,0.000000
25%,9.000000,1.000000
50%,12.000000,6.000000
75%,19.000000,34.000000
max,4054.000000,21218.000000


## Step 5 · Write cleaned extract + data-quality report

`events_clean.parquet` is still private, user-level data -- same gitignored tier as
the raw extract, just cleaned. The report is aggregate pipeline-health numbers only.


In [8]:
out_path = RAW_DIR / "events_clean.parquet"
events_clean.to_parquet(out_path, index=False)
print(f"Wrote {len(events_clean):,} rows -> {out_path.resolve()}")
print("Reminder: data/raw/ is gitignored. This file stays private.")

report = f"""# Data Quality Report -- 01_extract -> 02_clean_audit

_Generated by `02_clean_audit.ipynb`. Pipeline-health metrics only -- no user-level
data or business figures below._

## Pipeline funnel

| Stage | Rows | Removed |
|---|---:|---:|
| Raw extract | {n_raw:,} | -- |
| After dedup | {n_deduped:,} | {n_raw - n_deduped:,} ({(n_raw - n_deduped) / n_raw:.2%}) |
| After bot/non-prod filtering | {n_clean:,} | {n_deduped - n_clean:,} ({(n_deduped - n_clean) / n_deduped:.2%}) |

## Traffic filtering breakdown

- Non-production rows removed: {n_non_prod:,} ({n_non_prod / before:.2%})
- Bot/AI-agent/automation rows removed: {n_bot:,} ({n_bot / before:.2%})
- is_bot / traffic_type disagreement: {n_mismatch:,} rows

## Sessions

- Rows with a session_id: {with_session:,} ({with_session / n_clean:.2%})
- Rows with no session context (server-side events): {n_clean - with_session:,}
- Total sessions: {n_sessions:,}
- Median events/session: {median_events_per_session:.1f}
- Median session duration: {median_session_seconds:.0f}s

## Known gaps

- No per-event unique id (`uuid`/`$insert_id`) was pulled in `01_extract.ipynb`, so
  dedup only catches exact duplicate rows across the columns we selected, not
  ingestion-level duplicates. Add one of those ids upstream if this becomes a
  concern.
"""

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = REPORTS_DIR / "data_quality.md"
report_path.write_text(report, encoding="utf-8")
print(f"Wrote data-quality report -> {report_path.resolve()}")

Wrote 103,992 rows -> C:\Users\Faith\Desktop\peras-analytics\peras-analytics\data\raw\events_clean.parquet
Reminder: data/raw/ is gitignored. This file stays private.
Wrote data-quality report -> C:\Users\Faith\Desktop\peras-analytics\peras-analytics\reports\data_quality.md


### Bonus: CSV export for ad hoc visualization

Same data as events_clean.parquet, just as CSV for quick charting (Excel, Tableau,
a plotting library) before 03_eda.ipynb formalizes the analysis. Still private,
user-level data -- lands in data/raw/ (gitignored, and *.csv is gitignored
repo-wide as a second safety net) alongside the parquet version, never committed.


In [9]:
csv_path = RAW_DIR / "events_clean.csv"
events_clean.to_csv(csv_path, index=False)
print(f"Wrote {len(events_clean):,} rows -> {csv_path.resolve()}")
print("Reminder: data/raw/ is gitignored (and *.csv is gitignored repo-wide too). This file stays private.")


Wrote 103,992 rows -> C:\Users\Faith\Desktop\peras-analytics\peras-analytics\data\raw\events_clean.csv
Reminder: data/raw/ is gitignored (and *.csv is gitignored repo-wide too). This file stays private.


---

## Next

→ **`03_eda.ipynb`** — funnels, retention cohorts, segmentation, and experiment
analysis on `events_clean.parquet`.
